## ۱) بررسی GPU

In [111]:
import torch
assert torch.cuda.is_available(), "Select Runtime > Change runtime type > T4 GPU first."
print(torch.cuda.get_device_name(0))


Tesla T4


## ۲) نصب کتابخانه‌ها

In [112]:
import subprocess, sys, os, glob, shutil, site

os.environ["USE_TF"] = "0"


subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "-q",
                 "diffusers", "transformers", "accelerate"])
for pkg in ("diffusers", "transformers", "accelerate"):
    for base in site.getsitepackages() + [site.getusersitepackages()]:
        for path_ in glob.glob(os.path.join(base, pkg)):
            shutil.rmtree(path_, ignore_errors=True)

subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "--no-cache-dir",
    "diffusers==0.37.0", "transformers==4.57.6", "accelerate==1.12.0"])
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
    "flask", "flask-cors", "pyngrok", "ultralytics", "segment-anything",
    "opencv-python-headless", "safetensors"])

subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "--no-deps", "simple-lama-inpainting"])


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.0/5.0 MB 21.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 80.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 380.9/380.9 kB 400.1 MB/s eta 0:00:00


0

## ۳) آماده‌سازی مسیرهای ذخیره‌سازی

In [113]:
import os, torch

CACHE_DIR = "/content/interior_v2/models" if os.path.isdir("/content") else "/kaggle/working/models"
OUTPUT_DIR = "/content/interior_v2/outputs" if os.path.isdir("/content") else "/kaggle/working/outputs"
os.makedirs(CACHE_DIR, exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)
print("CACHE_DIR:", CACHE_DIR)


CACHE_DIR: /content/interior_v2/models


## ۴) بارگذاری مدل‌ها — دو مدل، لود تنبل



In [114]:
import gc
from diffusers import (
    ControlNetModel, AutoencoderKL, UniPCMultistepScheduler,
    StableDiffusionControlNetImg2ImgPipeline, StableDiffusionInpaintPipeline,
    StableDiffusionXLControlNetImg2ImgPipeline, AutoPipelineForInpainting,
)

style_pipe = None
inpaint_pipe = None
_current_model = {"name": None, "need": None}

GEN_PARAMS = {
    "fast":    {"guidance_scale": 9.0, "controlnet_conditioning_scale": 1.05},
    "quality": {"guidance_scale": 7.0, "controlnet_conditioning_scale": 0.5},
}


SD15_CHECKPOINT = "SG161222/Realistic_Vision_V6.0_B1_noVAE"
SD15_VAE = "stabilityai/sd-vae-ft-mse"
SDXL_CHECKPOINT = "RunDiffusion/Juggernaut-XL-v9"


def _load_sd15_style():
    global style_pipe
    print(f"Loading {SD15_CHECKPOINT} (Fast mode, style)...")
    cn = ControlNetModel.from_pretrained(
        "lllyasviel/control_v11p_sd15_canny", torch_dtype=torch.float16, cache_dir=CACHE_DIR
    )
    vae15 = AutoencoderKL.from_pretrained(SD15_VAE, torch_dtype=torch.float16, cache_dir=CACHE_DIR)
    
    style_pipe = StableDiffusionControlNetImg2ImgPipeline.from_pretrained(
        SD15_CHECKPOINT, controlnet=cn, vae=vae15, torch_dtype=torch.float16,
        safety_checker=None, cache_dir=CACHE_DIR
    )
    style_pipe.scheduler = UniPCMultistepScheduler.from_config(style_pipe.scheduler.config)
    style_pipe.load_ip_adapter("h94/IP-Adapter", subfolder="models", weight_name="ip-adapter-plus_sd15.bin")  # Plus: patch embeddings, حفظ جزئیات مرجع بهتر از نسخهٔ پایه
    style_pipe.set_ip_adapter_scale(0.0)
    style_pipe.enable_model_cpu_offload()
    style_pipe.enable_vae_tiling()
    print(f" {SD15_CHECKPOINT} ready (fast, photorealism-tuned, Img2Img)")


def _load_sd15_inpaint():
    global inpaint_pipe
    print("Loading SD1.5 inpainting (Fast mode)...")
    inpaint_pipe = StableDiffusionInpaintPipeline.from_pretrained(
        "stable-diffusion-v1-5/stable-diffusion-inpainting",
        torch_dtype=torch.float16, safety_checker=None, cache_dir=CACHE_DIR
    )
    inpaint_pipe.enable_model_cpu_offload()
    print(" SD1.5 inpainting ready")


def _load_sdxl_style():
    global style_pipe
    print(f"Loading {SDXL_CHECKPOINT} (Quality mode, style)... (bigger — first time downloads more)")
    cn = ControlNetModel.from_pretrained(
        "diffusers/controlnet-canny-sdxl-1.0", torch_dtype=torch.float16, cache_dir=CACHE_DIR,
        variant="fp16", use_safetensors=True,
    )
    vae = AutoencoderKL.from_pretrained(
        "madebyollin/sdxl-vae-fp16-fix", torch_dtype=torch.float16, cache_dir=CACHE_DIR
    )
    
    from transformers import CLIPVisionModelWithProjection
    image_encoder = CLIPVisionModelWithProjection.from_pretrained(
        "h94/IP-Adapter", subfolder="models/image_encoder", torch_dtype=torch.float16, cache_dir=CACHE_DIR,
    )
    style_pipe = StableDiffusionXLControlNetImg2ImgPipeline.from_pretrained(
        SDXL_CHECKPOINT, controlnet=cn, vae=vae, image_encoder=image_encoder, torch_dtype=torch.float16, cache_dir=CACHE_DIR,
        variant="fp16", use_safetensors=True,
    )
    style_pipe.scheduler = UniPCMultistepScheduler.from_config(style_pipe.scheduler.config)
    
    style_pipe.load_ip_adapter("h94/IP-Adapter", subfolder="sdxl_models", weight_name="ip-adapter-plus_sdxl_vit-h.safetensors")  # Plus: حفظ جزئیات مرجع بهتر از نسخهٔ پایه
    style_pipe.set_ip_adapter_scale(0.0)
    style_pipe.enable_model_cpu_offload()
    style_pipe.enable_vae_tiling()
    print(f" {SDXL_CHECKPOINT} ready (slower, best photorealism, Img2Img)")


def _load_sdxl_inpaint():
    global inpaint_pipe
    print("Loading SDXL inpainting (Quality mode)...")
   
    inpaint_pipe = AutoPipelineForInpainting.from_pretrained(
        "diffusers/stable-diffusion-xl-1.0-inpainting-0.1",
        torch_dtype=torch.float16, cache_dir=CACHE_DIR,
        variant="fp16", use_safetensors=True,
    )
    inpaint_pipe.enable_model_cpu_offload()
    print("SDXL inpainting ready")


def _unload_all():
    global style_pipe, inpaint_pipe
    if style_pipe is not None:
        style_pipe.to("cpu")
        del style_pipe
        style_pipe = None
    if inpaint_pipe is not None:
        inpaint_pipe.to("cpu")
        del inpaint_pipe
        inpaint_pipe = None
    gc.collect()
    torch.cuda.empty_cache()



def ensure_model(name, need="style"):
    """name: "fast" (Realistic Vision / SD1.5) or "quality" (Juggernaut XL / SDXL).
    need: "style" (generate_style) or "inpaint" (edit_object,
    delete/furnish/texture operations). Only ever ONE pipeline is resident at a time."""
    global style_pipe, inpaint_pipe
    name = name if name in ("fast", "quality") else "fast"
    need = need if need in ("style", "inpaint") else "style"
    if _current_model["name"] == name and _current_model["need"] == need:
        return
    if _current_model["name"] is not None:
        print(f"Switching model: {_current_model['name']}/{_current_model['need']} -> {name}/{need} (unloading previous one)...")
    _unload_all()
    if need == "style":
        (_load_sdxl_style if name == "quality" else _load_sd15_style)()
    else:
        (_load_sdxl_inpaint if name == "quality" else _load_sd15_inpaint)()
    _current_model["name"] = name
    _current_model["need"] = need


print(" Model loader ready — ensure_model('fast'/'quality', need='style'/'inpaint')")


 Model loader ready — ensure_model('fast'/'quality', need='style'/'inpaint')


In [115]:
ensure_model("fast", need="style")


Loading SG161222/Realistic_Vision_V6.0_B1_noVAE (Fast mode, style)...


Loading pipeline components...:   0%|          | 0/6 [00:00<?, ?it/s]

An error occurred while trying to fetch /content/interior_v2/models/models--SG161222--Realistic_Vision_V6.0_B1_noVAE/snapshots/9a857a696b9aabbf509073e0aa55ec8200b6ef7d/unet: Error no file named diffusion_pytorch_model.safetensors found in directory /content/interior_v2/models/models--SG161222--Realistic_Vision_V6.0_B1_noVAE/snapshots/9a857a696b9aabbf509073e0aa55ec8200b6ef7d/unet.
Defaulting to unsafe serialization. Pass `allow_pickle=False` to raise an error instead.
CLIPFeatureExtractor appears to have been deprecated in transformers. Using CLIPImageProcessor instead.
You have disabled the safety checker for <class 'diffusers.pipelines.controlnet.pipeline_controlnet_img2img.StableDiffusionControlNetImg2ImgPipeline'> by passing `safety_checker=None`. Ensure that you abide to the conditions of the Stable Diffusion license and do not expose unfiltered results in services or applications open to the public. Both the diffusers team and Hugging Face strongly recommend to keep the safety fil

 SG161222/Realistic_Vision_V6.0_B1_noVAE ready (fast, photorealism-tuned, Img2Img)


/usr/local/lib/python3.12/dist-packages/diffusers/pipelines/pipeline_utils.py:2290: FutureWarning: `enable_vae_tiling` is deprecated and will be removed in version 0.40.0. Calling `enable_vae_tiling()` on a `StableDiffusionControlNetImg2ImgPipeline` is deprecated and this method will be removed in a future version. Please use `pipe.vae.enable_tiling()`.
  deprecate(


## ۵) تشخیص اشیا — YOLOv8 + SAM

In [116]:
from ultralytics import YOLO

yolo_model = YOLO("yolov8x.pt")
yolo_model.to("cpu")
print(" YOLOv8 loaded (CPU)!")


 YOLOv8 loaded (CPU)!


In [117]:
from segment_anything import sam_model_registry, SamPredictor
import requests

sam_path = os.path.join(CACHE_DIR, "sam_vit_h.pth")

if not os.path.exists(sam_path):
    print("Downloading SAM model... (2-3 minutes)")
    url = "https://dl.fbaipublicfiles.com/segment_anything/sam_vit_h_4b8939.pth"
    response = requests.get(url, stream=True, timeout=120)
    response.raise_for_status()
    with open(sam_path + ".part", "wb") as f:
        for chunk in response.iter_content(chunk_size=8192):
            f.write(chunk)
    os.replace(sam_path + ".part", sam_path)
    response.close()
    print(" SAM downloaded")
else:
    print(" SAM already exists")

sam = sam_model_registry["vit_h"](checkpoint=sam_path)
sam.to("cpu")  
sam_predictor = SamPredictor(sam)
print("SAM loaded (CPU)")


 SAM already exists
SAM loaded (CPU)


## ۶) توابع کمکی — تبدیل تصویر، انتخاب ناحیه، inpaint موضعی


In [118]:
import base64, io, hashlib, threading, uuid as _uuid
from collections import OrderedDict
import numpy as np, cv2
from PIL import Image, ImageOps

MODEL_LOCK = threading.RLock()
IMAGE_STORE, REGION_STORE = OrderedDict(), OrderedDict()
MAX_IMAGES, MAX_REGION_SETS = 16, 8


def _decode_image(value):
    if not isinstance(value, str) or not value:
        raise ValueError("image must be base64")
    try:
        raw = base64.b64decode(value.split(",", 1)[-1], validate=True)
    except Exception as exc:
        raise ValueError("Invalid base64 image data") from exc
    try:
        im = Image.open(io.BytesIO(raw))
        im.load()  # چک‌های decode واقعی اینجا انجام می‌شه، نه در open() تنبل
    except Exception as exc:
        # علت رایج: HEIC (پیش‌فرض دوربین آیفون) — Pillow بدون افزونه اضافه نمی‌تونه بازش کنه.
        raise ValueError("Could not read this file as an image; use JPEG, PNG or WEBP (not HEIC/HEIF)") from exc
    if im.width * im.height > 16_000_000:
        raise ValueError(
            f"Image is {im.width}x{im.height} ({im.width * im.height / 1_000_000:.1f} megapixels); "
            "max is 16 megapixels — a modern phone photo can exceed this, resize it first"
        )
    return ImageOps.exif_transpose(im)


def base64_to_pil(value):
    return _decode_image(value).convert("RGB")


def base64_to_pil_rgba(value):
    return _decode_image(value).convert("RGBA")


def pil_to_base64(im):
    b = io.BytesIO()
    im.save(b, format="PNG")
    return base64.b64encode(b.getvalue()).decode()


def image_key(im):
    return hashlib.sha256(str(im.size).encode() + im.tobytes()).hexdigest()


def remember_image(im):
    key = _uuid.uuid4().hex
    IMAGE_STORE[key] = pil_to_base64(im)
    while len(IMAGE_STORE) > MAX_IMAGES:
        IMAGE_STORE.popitem(last=False)
    return {"image_id": key, "image": IMAGE_STORE[key], "mime_type": "image/png",
            "width": im.width, "height": im.height}


def processing_image(im):
    scale = 512 / max(im.size)
    size = tuple(max(1, round(v * scale)) for v in im.size)
    small = im.resize(size, Image.Resampling.LANCZOS)
    canvas = Image.new("RGB", (512, 512))
    canvas.paste(small, (0, 0))
    a = np.array(canvas)
    if size[0] < 512:
        a[:, size[0]:] = a[:, size[0] - 1:size[0]]
    if size[1] < 512:
        a[size[1]:] = a[size[1] - 1:size[1]]
    return Image.fromarray(a), size


def get_canny_edges(im):
    e = cv2.Canny(cv2.cvtColor(np.array(im), cv2.COLOR_RGB2GRAY), 100, 200)
    return Image.fromarray(np.repeat(e[..., None], 3, 2))


def text_prompt(v, name="prompt"):
    if not isinstance(v, str) or not v.strip():
        raise ValueError(name + " is required")
    if len(v) > 600:
        raise ValueError(name + " must be <= 600 characters")
    return v.strip()


def box_pixels(box, im):
    if not isinstance(box, list) or len(box) != 4:
        raise ValueError("bbox must be normalized [x1,y1,x2,y2]")
    x1, y1, x2, y2 = map(float, box)
    if not 0 <= x1 < x2 <= 1 or not 0 <= y1 < y2 <= 1:
        raise ValueError("bbox coordinates must be 0..1")
    return [int(x1 * im.width), int(y1 * im.height),
            min(im.width, int(np.ceil(x2 * im.width))),
            min(im.height, int(np.ceil(y2 * im.height)))]


def _points(values, im):
    if not isinstance(values, list) or not values:
        raise ValueError("positive points required")
    out = []
    for p in values:
        if (not isinstance(p, list) or len(p) != 2
                or not all(isinstance(x, (int, float)) and 0 <= x <= 1 for x in p)):
            raise ValueError("points must be normalized [x,y]")
        out.append([min(im.width - 1, p[0] * im.width), min(im.height - 1, p[1] * im.height)])
    return out


def sam_points_mask(im, positive, negative=None, bbox=None):
    pos = _points(positive, im)
    neg = [] if not negative else _points(negative, im)
    sam_predictor.set_image(np.array(im))
    m, s, _ = sam_predictor.predict(
        point_coords=np.array(pos + neg, np.float32),
        point_labels=np.array([1] * len(pos) + [0] * len(neg)),
        box=np.array(box_pixels(bbox, im), np.float32) if bbox else None,
        multimask_output=True,
    )
    mask = m[int(np.argmax(s))]
    dilate_px = max(2, round(min(im.size) * 0.004))
    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (2 * dilate_px + 1, 2 * dilate_px + 1))
    return cv2.dilate(mask.astype("uint8"), kernel, iterations=1).astype(bool)


def selection_mask(im, selection):
    if not isinstance(selection, dict):
        raise ValueError("selection is required")
    kinds = [k for k in ("region_id", "mask", "bbox", "point", "points") if selection.get(k) is not None]
    if len(kinds) != 1:
        raise ValueError("Choose one selection type")
    k = kinds[0]
    if k == "region_id":
        item = REGION_STORE.get(image_key(im), {}).get(selection[k])
        if item is None:
            raise ValueError("Region expired; use its mask or detect again")
        mask = item["mask"].copy()
    elif k == "mask":
        v = base64_to_pil(selection[k]).convert("L")
        if v.size != im.size:
            raise ValueError("mask dimensions must match image")
        mask = np.array(v) >= 128
    elif k == "bbox":
        x1, y1, x2, y2 = box_pixels(selection[k], im)
        mask = np.zeros((im.height, im.width), bool)
        mask[y1:y2, x1:x2] = 1
    elif k == "point":
        mask = sam_points_mask(im, [selection[k]])
    else:
        spec = selection[k]
        if not isinstance(spec, dict):
            raise ValueError("points must be an object")
        mask = sam_points_mask(im, spec.get("positive"), spec.get("negative"), spec.get("bbox"))
    if not mask.any():
        raise ValueError("Selection is empty")
    return mask


NEGATIVE = "blurry, distorted, watermark, changed architecture, extra objects"


def localized_inpaint(im, mask, prompt, seed=42, negative=NEGATIVE, expand_px=10, feather_px=5, strength=.98):
    raw = mask.astype("uint8") * 255
    if expand_px:
        k = 2 * expand_px + 1
        raw = cv2.dilate(raw, np.ones((k, k), np.uint8))
    ys, xs = np.where(raw > 0)
    if not len(xs):
        raise ValueError("Selection is empty")
    x1, x2, y1, y2 = xs.min(), xs.max() + 1, ys.min(), ys.max() + 1
    span = max(x2 - x1, y2 - y1)
    pad = max(24, int(span * .38))
    cx, cy = (x1 + x2) // 2, (y1 + y2) // 2
    side = max(128, span + 2 * pad)
    left = max(0, cx - side // 2)
    top = max(0, cy - side // 2)
    right = min(im.width, left + side)
    bottom = min(im.height, top + side)
    left = max(0, right - side)
    top = max(0, bottom - side)

    crop = im.crop((left, top, right, bottom))
    mc = Image.fromarray(raw).crop((left, top, right, bottom))
    canvas, size = processing_image(crop)

    mask512 = Image.new("L", (512, 512))
    mask512.paste(mc.resize(size, Image.Resampling.NEAREST), (0, 0))

    out = inpaint_pipe(
        prompt=prompt, negative_prompt=negative, image=canvas, mask_image=mask512,
        height=512, width=512, strength=strength, num_inference_steps=45, guidance_scale=9,
        generator=torch.Generator(device="cuda").manual_seed(seed),
    ).images[0]

    gen = out.crop((0, 0, *size)).resize(crop.size, Image.Resampling.LANCZOS)
    alpha = np.array(mc)
    if feather_px:
        alpha = cv2.GaussianBlur(alpha, (0, 0), feather_px / 2)

    final = im.copy()
    final.paste(Image.composite(gen, crop, Image.fromarray(alpha)), (left, top))
    return final


print(" Helper functions ready ")


 Helper functions ready 


## ۷) پرامپت‌های ۸ سبک طراحی

In [119]:
STYLE_PROMPTS = {
    "minimalist": {
        "prompt": "minimalist interior design, pure white and warm beige walls, polished concrete or light wood floor, clean geometric lines, no clutter, abundant natural daylight, recessed ceiling lights, hidden storage",
        "negative": "cluttered, colorful, busy, ornate, dark, multiple patterns, too many objects"
    },
    "industrial": {
        "prompt": "industrial interior design, raw exposed red brick walls, polished concrete floor, black steel window frames, exposed metal ceiling beams, Edison bulb pendant lights, reclaimed dark wood and iron furniture",
        "negative": "soft, pastel, floral, traditional, plastic, bright white, fancy"
    },
    "cyberpunk": {
        "prompt": "cyberpunk interior design, dark charcoal walls with hexagonal panels, RGB LED strip lighting glowing blue and purple, neon pink and cyan signs on wall, carbon fiber furniture, city skyline through window at night with rain",
        "negative": "foggy, hazy, too dark, traditional, wooden, daytime, bright white, plain walls"
    },
    "modern_luxury": {
        "prompt": "ultra luxury modern interior design, Calacatta marble accent wall with gold veining, herringbone light oak floor, cream boucle upholstered furniture, sculptural gold brass chandelier, floor to ceiling ivory silk curtains",
        "negative": "cheap, basic, plastic, clutter, industrial, rustic, dark, crowded"
    },
    "scandinavian": {
        "prompt": "Scandinavian hygge interior design, white washed pine plank floors, pale sage green accent wall, natural oak surfaces, soft wool throws in oatmeal color, rattan pendant light, wall-mounted oak shelves with ceramic vases",
        "negative": "dark, ornate, cluttered, colorful, heavy patterns, gilded, industrial, cold"
    },
    "midcentury_modern": {
        "prompt": "mid century modern interior design, warm walnut teak wood furniture, geometric patterned wool rug in orange and brown, Eames style lounge chair, tulip side table, sunburst wall clock in gold, tapered furniture legs",
        "negative": "contemporary, futuristic, traditional, ornate, dark, gothic, cold colors"
    },
    "japanese_zen": {
        "prompt": "Japanese zen interior design, natural tatami-textured flooring, shoji screen panels with warm backlight, unfinished hinoki wood walls, bamboo ceiling accents, bonsai tree on wooden stand, earthy sand beige and forest green tones",
        "negative": "cluttered, colorful, western, modern tech, busy patterns, gold, ornate"
    },
    "bohemian": {
        "prompt": "bohemian interior design, terracotta painted walls, layered Persian and Moroccan rugs on wooden floor, macrame wall hanging, rattan egg chair suspended from a visible ceiling chain, trailing plants in woven pots, warm string fairy lights",
        "negative": "minimal, plain, cold, sterile, corporate, modern sleek, industrial, sparse"
    },
    "art_deco": {
        "prompt": "art deco interior design, geometric sunburst wall panel in black lacquer and gold trim, emerald green velvet armchair, brass and marble side table, fluted glass pendant chandelier, herringbone parquet floor, curved furniture silhouettes",
        "negative": "rustic, farmhouse, plain wood, minimalist, cheap plastic, flat lighting, dull colors"
    },
    "coastal": {
        "prompt": "coastal interior design, whitewashed shiplap walls, light bleached oak floor, linen slipcovered sofa in soft white, jute rope rug, driftwood accent table, sheer breezy curtains, soft blue and sandy beige palette",
        "negative": "dark, heavy, industrial, cluttered, neon, gothic, ornate gold, cramped"
    },
    "french_country": {
        "prompt": "French country interior design, soft cream limewashed walls, exposed wooden ceiling beams, toile de Jouy upholstered armchair, wrought iron chandelier, weathered oak farmhouse table, powder blue painted cabinetry, terracotta tile floor",
        "negative": "modern minimalist, industrial, neon, plastic, sterile, cold lighting"
    },
    "farmhouse_rustic": {
        "prompt": "rustic farmhouse interior design, reclaimed barn wood accent wall, wide plank distressed hardwood floor, black iron light fixtures, open wooden shelving with ceramic crockery, stone fireplace surround, oversized linen sofa, warm cream and charcoal palette",
        "negative": "glossy, futuristic, neon, cyberpunk, sterile, corporate, plastic"
    },
    "contemporary_glam": {
        "prompt": "contemporary glam interior design, plush tufted velvet sofa in blush pink, polished chrome and lucite coffee table, oversized crystal chandelier, metallic gold throw pillows, high gloss lacquer cabinetry, faux fur area rug, glossy marble floor",
        "negative": "rustic, farmhouse, raw concrete, dull, matte flat colors, cheap, worn"
    },
    "dark_academia": {
        "prompt": "dark academia interior design, floor to ceiling dark walnut bookshelves with leather bound books, deep forest green velvet wingback armchair, brass library lamp with green glass shade, oxblood leather chesterfield sofa, herringbone dark wood floor",
        "negative": "bright white, minimalist, neon, plastic, modern sleek, sparse, empty shelves"
    },
    "tropical_modern": {
        "prompt": "tropical modern interior design, natural rattan and woven wicker furniture, abundant monstera and palm plants, warm teak wood accents, open airy layout with louvered shutters, terracotta and cream palette, woven bamboo pendant light",
        "negative": "cold, sterile, heavy dark wood, no plants, industrial, gothic, cramped"
    },
    "brutalist": {
        "prompt": "brutalist interior design, raw exposed poured concrete walls and ceiling, monolithic geometric furniture forms, single sculptural black leather chair, minimal charcoal grey palette, exposed structural beams, stark dramatic shadows",
        "negative": "ornate, colorful, cluttered, cozy textiles, floral, pastel, cheap plastic"
    },
    "mediterranean": {
        "prompt": "Mediterranean coastal interior design, whitewashed stucco walls, blue and white azulejo tile accents, wrought iron light fixtures, terracotta floor tiles, olive green potted plants, arched doorways",
        "negative": "cold, minimalist, industrial, neon, dark, plastic, sterile"
    },
    "shabby_chic": {
        "prompt": "shabby chic interior design, distressed white painted furniture, soft pastel floral upholstery, vintage crystal chandelier, weathered wood accents, lace and linen textiles, romantic vintage charm",
        "negative": "modern sleek, industrial, dark colors, minimalist, glossy plastic, bold geometric"
    },
    "southwestern_desert": {
        "prompt": "southwestern desert interior design, warm adobe clay walls, Navajo pattern woven textiles, turquoise accent pillows, leather furniture, cacti in terracotta pots, natural wood beams",
        "negative": "cold, pastel, industrial, glossy, minimalist white, nautical"
    },
    "memphis_postmodern": {
        "prompt": "Memphis postmodern interior design, bold primary color blocks, squiggle and geometric patterns, playful asymmetric furniture shapes, terrazzo flooring, chrome and neon accents",
        "negative": "muted, traditional, rustic, minimalist, dark academia, farmhouse"
    }
}

ROOM_PRESERVE_LEAD = "keep the same room type, walls, windows, doors and furniture layout"
ROOM_PRESERVE_NEGATIVE = ", different room type, structural changes, moved walls or windows"

ARTIFACT_NEGATIVE_LEAD = "floating furniture, furniture not touching the floor, duplicate chairs, cloned furniture, stains, blemishes"

QUALITY_SUFFIX = ", RAW photo, professional real estate photography"
QUALITY_NEGATIVE = ", illustration, painting, cartoon, 3d render, cgi"
TAIL_NEGATIVE = ", blurry, low quality, watermark"
REFERENCE_ROOM_IP_SCALE = 0.5

print("Style prompts ready:", list(STYLE_PROMPTS.keys()))


Style prompts ready: ['minimalist', 'industrial', 'cyberpunk', 'modern_luxury', 'scandinavian', 'midcentury_modern', 'japanese_zen', 'bohemian', 'art_deco', 'coastal', 'french_country', 'farmhouse_rustic', 'contemporary_glam', 'dark_academia', 'tropical_modern', 'brutalist', 'mediterranean', 'shabby_chic', 'southwestern_desert', 'memphis_postmodern']


## ۸) تشخیص اشیا — YOLO + SAM + SegFormer


In [120]:
_semantic_processor = _semantic_model = None

def semantic_regions(im, min_confidence=0.45, max_side=1024):
    global _semantic_processor, _semantic_model
    if _semantic_model is None:
        from transformers import AutoImageProcessor, SegformerForSemanticSegmentation
        
        model_id = "nvidia/segformer-b2-finetuned-ade-512-512"
        _semantic_processor = AutoImageProcessor.from_pretrained(model_id, cache_dir=CACHE_DIR)
        _semantic_model = SegformerForSemanticSegmentation.from_pretrained(model_id, cache_dir=CACHE_DIR).eval()

    inputs = _semantic_processor(images=im, return_tensors="pt")
    with torch.inference_mode():
        output = _semantic_model(**inputs)

    scale = min(1.0, max_side / max(im.size))
    work_w, work_h = (max(1, round(im.width * scale)), max(1, round(im.height * scale))) if scale < 1 else im.size
    upsampled = torch.nn.functional.interpolate(
        output.logits, size=(work_h, work_w), mode="bilinear", align_corners=False
    )
    confidence, labels = upsampled.softmax(dim=1)[0].max(dim=0)
    labels = labels.cpu().numpy()
    confidence = confidence.cpu().numpy()
   
    min_area = max(64 * scale * scale, work_w * work_h * 0.0012)

    mirror_mask = np.zeros(labels.shape, dtype=bool)
    for cls in np.unique(labels):
        if _semantic_model.config.id2label[int(cls)].split(";")[0].lower() == "mirror":
            mirror_mask |= labels == cls

    for cls in np.unique(labels):
        label = _semantic_model.config.id2label[int(cls)].split(";")[0]
        is_mirror = label.lower() == "mirror"
        n, parts, stats, _ = cv2.connectedComponentsWithStats((labels == cls).astype("uint8"), 8)
        for i in range(1, n):
            if stats[i, cv2.CC_STAT_AREA] < min_area:
                continue
            component = parts == i
            if confidence[component].mean() < min_confidence:
                continue
            if not is_mirror:
                component = component & ~mirror_mask
                if component.sum() < min_area:
                    continue
            if scale < 1:
                component = cv2.resize(component.astype("float32"), im.size, interpolation=cv2.INTER_LINEAR) >= 0.5
            yield label, component


def detect_objects(image_b64):
    im = base64_to_pil(image_b64)
    key, items, public = image_key(im), {}, []
    dilate_px = max(2, round(min(im.size) * 0.004))
    dilate_kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (2 * dilate_px + 1, 2 * dilate_px + 1))

    def add(label, mask, source, confidence=None):
        if len(items) >= 80 or not mask.any():
            return
        mask = cv2.dilate(mask.astype("uint8"), dilate_kernel, iterations=1).astype(bool)
        ys, xs = np.where(mask)
        rid = _uuid.uuid4().hex
        items[rid] = {"label": label, "mask": mask, "confidence": confidence}
        public.append({
            "id": rid, "label": label, "source": source, "confidence": confidence,
            "bbox": [float(xs.min() / im.width), float(ys.min() / im.height),
                     float((xs.max() + 1) / im.width), float((ys.max() + 1) / im.height)],
            "mask": pil_to_base64(Image.fromarray(mask.astype("uint8") * 255)),
            "mask_mime_type": "image/png",
        })


    SAM_MIN_SCORE = 0.5
    YOLO_IOU = 0.5

    DUPLICATE_IOU = 0.7

    with MODEL_LOCK:
        sam_predictor.set_image(np.array(im))
        for result in yolo_model(np.array(im), verbose=False, conf=0.2, iou=YOLO_IOU, max_det=40):
            for box in result.boxes:
                masks, scores, _ = sam_predictor.predict(box=box.xyxy[0].cpu().numpy(), multimask_output=True)
                best = int(np.argmax(scores))
                if scores[best] < SAM_MIN_SCORE:
                    continue  # SAM خودش به هیچ‌کدوم از ۳ کاندیدش مطمئن نبود -- سگمنت الکی رد می‌شه
                mask, conf = masks[best], float(box.conf.item())
                if any((mask & item["mask"]).sum() / max(1, (mask | item["mask"]).sum()) > DUPLICATE_IOU
                       and conf <= (item.get("confidence") or 0) for item in items.values()):
                    continue
                add(yolo_model.names[int(box.cls.item())], mask, "yolo_sam", conf)

        object_mask = np.zeros((im.height, im.width), bool)
        for item in items.values():
            object_mask |= item["mask"]
        for label, mask in semantic_regions(im):
            if any(label == item["label"] and (mask & item["mask"]).sum() / max(1, (mask | item["mask"]).sum()) > 0.65
                   for item in items.values()):
                continue
            mask = mask & ~object_mask
            if not mask.any():
                continue
            add(label, mask, "segformer")
        REGION_STORE[key] = items
        while len(REGION_STORE) > MAX_REGION_SETS:
            REGION_STORE.popitem(last=False)

    return {"objects": list(dict.fromkeys(x["label"] for x in public)), "regions": public,
            "width": im.width, "height": im.height, "coordinates": "normalized", "image_hash": key}


print("detect_objects ready (YOLO + SAM + SegFormer)")


detect_objects ready (YOLO + SAM + SegFormer)


## ۹) تولید سبک — Img2Img از روی عکس واقعی

In [121]:
_IP_ADAPTER_NEUTRAL_IMAGE = Image.new("RGB", (224, 224), (128, 128, 128))


def generate_style(image_b64, style_name=None, palette=None, custom_prompt=None, colors_only=False,
                    extra_details=None, reference_image=None, model="fast", seed=42, draft=False):
    ensure_model(model, need="style")
    params = GEN_PARAMS.get(model, GEN_PARAMS["fast"])

    strength = 0.7

    steps = 12 if draft else 40
    canvas_size = 384 if draft else 768


    if reference_image:

        p = ROOM_PRESERVE_LEAD + ", match the overall design style, color palette, materials and furniture character of the reference image"
        neg = ARTIFACT_NEGATIVE_LEAD + ROOM_PRESERVE_NEGATIVE
    elif colors_only:
        if not palette:
            raise ValueError("colorsOnly requires a palette")
        p = ROOM_PRESERVE_LEAD + ", only repaint the walls and change decor colors to match this palette"
        neg = "changed furniture, changed layout, new objects, removed objects" + ROOM_PRESERVE_NEGATIVE
        strength = 0.4  # much less freedom than a full style change — colors only
    elif custom_prompt:
        p = ROOM_PRESERVE_LEAD + ", " + text_prompt(custom_prompt, "customPrompt")
        neg = ARTIFACT_NEGATIVE_LEAD + ROOM_PRESERVE_NEGATIVE
    elif style_name in STYLE_PROMPTS:
        config = STYLE_PROMPTS[style_name]
        base = config["prompt"]
        if extra_details:
            base += ", " + text_prompt(extra_details, "extraDetails")
        p = ROOM_PRESERVE_LEAD + ", " + base
        neg = ARTIFACT_NEGATIVE_LEAD + ", " + config["negative"] + ROOM_PRESERVE_NEGATIVE
    else:
        raise ValueError("Choose a valid style, customPrompt, colorsOnly, or referenceImage")

    if palette:
        if not isinstance(palette, dict):
            raise ValueError("palette must be an object")
        palette_prompt = palette.get("prompt", "")
        colors = palette.get("colors", [])
        if palette_prompt and not isinstance(palette_prompt, str):
            raise ValueError("palette.prompt must be text")
        if colors and (not isinstance(colors, list) or not all(isinstance(c, str) for c in colors)):
            raise ValueError("palette.colors must be a list")
        color_text = ", ".join(colors)
        palette_text = ", ".join(x for x in [palette_prompt.strip(), color_text] if x)
        if palette_text:
            p = palette_text + ", " + p + ", dominant color scheme must follow the selected palette"
            neg += ", wrong colors, different color palette, ignore selected palette"

    original = base64_to_pil(image_b64)
    size = original.size
    room = original.resize((canvas_size, canvas_size), Image.Resampling.LANCZOS)
    edge_image = get_canny_edges(room)
    ref_pil = base64_to_pil(reference_image) if reference_image else None

    with MODEL_LOCK:
        style_pipe.set_ip_adapter_scale(REFERENCE_ROOM_IP_SCALE if ref_pil else 0.0)
        try:
            r = style_pipe(
                prompt=p + QUALITY_SUFFIX,
                negative_prompt=neg + QUALITY_NEGATIVE + TAIL_NEGATIVE,
                image=room,               # عکس واقعی اتاق — Img2Img از پیکسل واقعی شروع می‌کنه
                control_image=edge_image,  # نقشهٔ لبه — ساختار رو هم‌راستا نگه می‌داره
                strength=strength,          # چقدر اجازه داره عکس واقعی تغییر کنه
                ip_adapter_image=ref_pil if ref_pil else _IP_ADAPTER_NEUTRAL_IMAGE,
                num_inference_steps=steps,
                guidance_scale=params["guidance_scale"],
                controlnet_conditioning_scale=params["controlnet_conditioning_scale"],
                height=canvas_size, width=canvas_size,
                generator=torch.Generator(device="cuda").manual_seed(seed),
            ).images[0]
        finally:
            if ref_pil:
                style_pipe.set_ip_adapter_scale(0.0)

    return {"image": pil_to_base64(r.resize(size, Image.Resampling.LANCZOS)), "mime_type": "image/png"}


def generate_all_previews(image_b64, palette=None, styles=None, model="fast", seed=42, draft=False):
    styles = list(STYLE_PROMPTS) if styles is None else styles
    n = len(STYLE_PROMPTS)
    if not isinstance(styles, list) or not 1 <= len(styles) <= n or any(s not in STYLE_PROMPTS for s in styles):
        raise ValueError(f"styles must contain 1..{n} valid names")
    return {
        "previews": {s: generate_style(image_b64, s, palette=palette, model=model, seed=seed, draft=draft)["image"] for s in styles},
        "mime_type": "image/png",
    }


print(" generate_style / generate_all_previews ready (model=\"fast\"/\"quality\")")


 generate_style / generate_all_previews ready (model="fast"/"quality")


## ۱۰) ویرایش، حذف و مبله‌کردن — انتخاب دقیق ناحیه


In [122]:
def edit_object(image_b64, object_label=None, edit_prompt=None, selection=None, model="fast", seed=42):
    ensure_model(model, need="inpaint")
    im = base64_to_pil(image_b64)
    with MODEL_LOCK:
        mask = selection_mask(im, selection)
        p = (text_prompt(edit_prompt) +
             ", blended naturally into the surrounding room, matching perspective, scale and existing lighting, photorealistic" + QUALITY_SUFFIX)
        neg = (ARTIFACT_NEGATIVE_LEAD +
               ", unrelated new object, extra furniture, mismatched style, warped lines, seams, bad edges" + QUALITY_NEGATIVE + TAIL_NEGATIVE)
        r = localized_inpaint(im, mask, p, seed, neg, 10, 5)
    return {"image": pil_to_base64(r), "mime_type": "image/png"}


_lama_model = None


def get_lama_model():
    global _lama_model
    if _lama_model is None:
        from simple_lama_inpainting import SimpleLama
        _lama_model = SimpleLama()
    return _lama_model




def expanded_delete_mask(mask, im):
    raw = mask.astype("uint8") * 255
    ys, xs = np.where(raw > 0)
    if not len(xs):
        raise ValueError("Selection is empty")
    object_span = max(xs.max() - xs.min() + 1, ys.max() - ys.min() + 1)
    radius = int(np.clip(round(object_span * .075), 8, max(12, round(min(im.size) * .06))))
    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (2 * radius + 1, 2 * radius + 1))
    raw = cv2.morphologyEx(raw, cv2.MORPH_CLOSE, kernel)
    raw = cv2.dilate(raw, kernel, iterations=1)
    return raw


def delete_object_lama(image_b64, selection):
    im = base64_to_pil(image_b64)
    with MODEL_LOCK:
        mask = selection_mask(im, selection)
        expanded = expanded_delete_mask(mask, im)
        result = get_lama_model()(im, Image.fromarray(expanded, mode="L")).convert("RGB")
        feather = max(2, round(min(im.size) * .006))
        alpha = cv2.GaussianBlur(expanded, (0, 0), feather)
        out = Image.composite(result, im, Image.fromarray(alpha))
    return {"image": pil_to_base64(out), "mime_type": "image/png"}


def recolor_object(image_b64, selection, color, strength=.85):
    if not isinstance(color, str) or len(color) != 7 or color[0] != "#":
        raise ValueError("color must be #RRGGBB")
    try:
        target_rgb = tuple(int(color[i:i + 2], 16) for i in (1, 3, 5))
    except ValueError as exc:
        raise ValueError("color must be #RRGGBB") from exc
    if not isinstance(strength, (int, float)) or not 0 <= float(strength) <= 1:
        raise ValueError("strength must be between 0 and 1")

    im = base64_to_pil(image_b64)
    with MODEL_LOCK:
        mask = selection_mask(im, selection)
        rgb = np.array(im)
        lab = cv2.cvtColor(rgb, cv2.COLOR_RGB2LAB).astype(np.float32)
        target = np.uint8([[target_rgb]])
        target_lab = cv2.cvtColor(target, cv2.COLOR_RGB2LAB)[0, 0].astype(np.float32)

        amount = float(strength)
        lab[..., 1] = lab[..., 1] * (1 - amount) + target_lab[1] * amount
        lab[..., 2] = lab[..., 2] * (1 - amount) + target_lab[2] * amount
        recolored = cv2.cvtColor(np.clip(lab, 0, 255).astype(np.uint8), cv2.COLOR_LAB2RGB)

        edge = max(1, round(min(im.size) * .004))
        alpha = cv2.GaussianBlur(mask.astype(np.float32), (0, 0), edge)[..., None]
        out = (recolored * alpha + rgb * (1 - alpha)).clip(0, 255).astype(np.uint8)
    return {"image": pil_to_base64(Image.fromarray(out)), "mime_type": "image/png"}


def apply_texture(image_b64, selection, texture_b64, opacity=.85):
    if not isinstance(opacity, (int, float)) or not 0 <= float(opacity) <= 1:
        raise ValueError("opacity must be between 0 and 1")
    im = base64_to_pil(image_b64)
    texture = base64_to_pil(texture_b64)

    with MODEL_LOCK:
        mask = selection_mask(im, selection)
        rgb = np.array(im).astype(np.float32)
        gray = cv2.cvtColor(np.array(im), cv2.COLOR_RGB2GRAY).astype(np.float32)

        ys, xs = np.where(mask)
        y0, y1, x0, x1 = ys.min(), ys.max() + 1, xs.min(), xs.max() + 1
        region_h, region_w = y1 - y0, x1 - x0

        tw, th = texture.size
        tiles_x = int(np.ceil(region_w / tw)) + 1
        tiles_y = int(np.ceil(region_h / th)) + 1
        tiled = Image.new("RGB", (tw * tiles_x, th * tiles_y))
        for ty in range(tiles_y):
            for tx in range(tiles_x):
                tiled.paste(texture, (tx * tw, ty * th))
        tiled_rgb = np.array(tiled.crop((0, 0, region_w, region_h))).astype(np.float32)

        local_gray = gray[y0:y1, x0:x1]
        local_mean = max(1.0, local_gray[mask[y0:y1, x0:x1]].mean())
        lighting = np.clip(local_gray / local_mean, .35, 1.8)[..., None]
        lit_texture = np.clip(tiled_rgb * lighting, 0, 255)

        out = rgb.copy()
        edge = max(1, round(min(im.size) * .004))
        region_mask = cv2.GaussianBlur(mask[y0:y1, x0:x1].astype(np.float32) * float(opacity), (0, 0), edge)[..., None]
        out[y0:y1, x0:x1] = lit_texture * region_mask + rgb[y0:y1, x0:x1] * (1 - region_mask)

    return {"image": pil_to_base64(Image.fromarray(np.clip(out, 0, 255).astype(np.uint8))), "mime_type": "image/png"}


TEXTURE_PROMPTS = {
    "natural_stone": {"category": "surface", "prompt": "natural stone cladding, cool grey and beige veined marble texture, polished stone surface, subtle natural veining pattern"},
    "wood_paneling": {"category": "surface", "prompt": "warm wood paneling, vertical oak wood slats, natural wood grain texture, honey brown tone"},
    "exposed_brick": {"category": "surface", "prompt": "exposed red brick wall, weathered brick texture, visible mortar lines, warm terracotta brick tones"},
    "exposed_concrete": {"category": "surface", "prompt": "raw exposed concrete surface, smooth grey concrete texture, subtle form-tie marks, industrial finish"},
    "geometric_wallpaper": {"category": "surface", "prompt": "geometric wallpaper pattern, repeating art deco gold and cream geometric print, elegant wall covering"},
    "ceramic_tile": {"category": "surface", "prompt": "glossy ceramic subway tile, clean white tile texture, thin grey grout lines, reflective glaze"},
    "leather": {"category": "furniture", "prompt": "smooth genuine leather upholstery, rich cognac brown leather grain, natural creases, stitched seams"},
    "velvet_fabric": {"category": "furniture", "prompt": "plush velvet fabric texture, soft deep emerald green velvet, rich fabric weave, luxurious upholstery texture"},
    "linen_fabric": {"category": "furniture", "prompt": "woven linen fabric upholstery, natural oatmeal linen weave texture, soft matte finish"},
    "suede": {"category": "furniture", "prompt": "soft suede upholstery texture, warm taupe suede nap, velvety matte finish"},
    "rattan_wicker": {"category": "furniture", "prompt": "natural rattan wicker weave texture, woven cane pattern, warm tan natural fiber texture"},
}



def generate_texture(image_b64, selection, texture_name, model="fast", seed=42):
    ensure_model(model, need="inpaint")
    if texture_name not in TEXTURE_PROMPTS:
        raise ValueError(f"texture must be one of {list(TEXTURE_PROMPTS)}")
    im = base64_to_pil(image_b64)
    with MODEL_LOCK:
        mask = selection_mask(im, selection)
        p = (TEXTURE_PROMPTS[texture_name]["prompt"] +
             ", seamless repeating texture, realistic material, matching existing lighting and shadows" + QUALITY_SUFFIX)
        neg = (ARTIFACT_NEGATIVE_LEAD + ", changed room style, changed furniture, different material, new object" +
               QUALITY_NEGATIVE + TAIL_NEGATIVE)
        r = localized_inpaint(im, mask, p, seed, neg, 6, 6, strength=.75)
    return {"image": pil_to_base64(r), "mime_type": "image/png"}


def resolve_selection(image_b64, data):
    """selection صریح رو برمی‌گردونه، یا اگه فقط اسم شیء (object) داده شده بود،
    یه تشخیص می‌زنه و اولین ناحیهٔ هم‌نام رو به یه region_id تبدیل می‌کنه —
    برای سازگاری با فرانتی که هنوز UI انتخاب ناحیه نداره."""
    selection = data.get("selection")
    if selection:
        return selection
    label = (data.get("object") or "").strip().lower()
    if not label:
        raise ValueError("selection or object is required")
    det = detect_objects(image_b64)
    match = next((r for r in det["regions"] if label in r["label"].lower()), None)
    if match is None:
        raise ValueError(f'No detected object matches "{label}"; try detect-objects first and pass a selection')
    return {"region_id": match["id"]}


def default_furnish_mask(im):
    m = np.zeros((im.height, im.width), bool)
    m[int(im.height * .43):, :] = 1
    return m


def furnish_room(image_b64, furnish_prompt, selection=None, model="fast", seed=42):
    ensure_model(model, need="inpaint")
    p = text_prompt(furnish_prompt)
    im = base64_to_pil(image_b64)
    with MODEL_LOCK:
        m = default_furnish_mask(im) if selection is None else selection_mask(im, selection)
        r = localized_inpaint(
            im, m,
            p + ", add only requested furniture, preserve existing room style, architecture and unrequested objects, realistic placement, perspective, scale and contact shadows" + QUALITY_SUFFIX,
            seed, "changed room style, changed walls, changed windows, duplicate furniture, floating object, blurry, watermark" + QUALITY_NEGATIVE, 4, 5,
        )
    return {"image": pil_to_base64(r), "mime_type": "image/png"}


def furnish_room_inpaint(image_b64, furnish_prompt, selection=None, model="fast", seed=42):
    return furnish_room(image_b64, furnish_prompt, selection, model, seed)


print(" edit_object / furnish_room / delete_object_lama / recolor_object ready")

 edit_object / furnish_room / delete_object_lama / recolor_object ready


## ۱۱) افزودن شیء از روی عکس مرجع — IP-Adapter


In [123]:
def _resize_to_fit(obj, box_w, box_h):
    box_w, box_h = max(1, box_w), max(1, box_h)
    obj_ratio, box_ratio = obj.width / obj.height, box_w / box_h
    if obj_ratio > box_ratio:
        new_w, new_h = box_w, max(1, round(box_w / obj_ratio))
    else:
        new_h, new_w = box_h, max(1, round(box_h * obj_ratio))
    return obj.resize((new_w, new_h), Image.Resampling.LANCZOS)



def place_library_object(room_image_b64, object_image_b64, placement_prompt, selection, model="fast", seed=42):
    ensure_model(model, need="inpaint")
    room = base64_to_pil(room_image_b64)
    obj = base64_to_pil_rgba(object_image_b64)

    footprint = selection_mask(room, selection)
    ys, xs = np.where(footprint)
    if not len(xs):
        raise ValueError("Selection is empty")
    x1, x2, y1, y2 = xs.min(), xs.max() + 1, ys.min(), ys.max() + 1
    fitted = _resize_to_fit(obj, x2 - x1, y2 - y1)

    px = x1 + (x2 - x1 - fitted.width) // 2
    py = y2 - fitted.height

    composited = room.convert("RGBA")
    composited.alpha_composite(fitted, (px, py))
    composited = composited.convert("RGB")

    obj_alpha = np.array(fitted.split()[-1])
    mask = np.zeros((room.height, room.width), dtype=np.uint8)
    mask[py:py + fitted.height, px:px + fitted.width] = obj_alpha
    dilate_px = max(8, round(min(room.size) * .015))
    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (2 * dilate_px + 1, 2 * dilate_px + 1))
    mask = cv2.dilate(mask, kernel, iterations=1) > 0

    p = ((text_prompt(placement_prompt) if placement_prompt.strip() else "the object rests naturally in the room") +
         ", realistic contact shadow, matching existing lighting and perspective, photorealistic" + QUALITY_SUFFIX)
    neg = (ARTIFACT_NEGATIVE_LEAD + ", moved object, different object, changed object shape, extra objects" +
           QUALITY_NEGATIVE + TAIL_NEGATIVE)

    with MODEL_LOCK:
        result = localized_inpaint(composited, mask, p, seed, neg, 4, 10, strength=.4)
    return {"image": pil_to_base64(result), "mime_type": "image/png"}


print("place_library_object ready (model=\"fast\"/\"quality\") — localized to the marked area when a selection is given")


place_library_object ready (model="fast"/"quality") — localized to the marked area when a selection is given


## ۱۲) تست  



In [124]:
from IPython.display import display
import matplotlib.pyplot as plt

TEST_STYLE = "minimalist"
TEST_MODEL = "fast"  


print("مرحله ۱: تو نوار سمت چپ Colab، روی آیکون پوشه 📁 کلیک کن.")
print("مرحله ۲: عکس اتاق رو از کامپیوترت بکش و همون‌جا ول کن (drag & drop) تا آپلود بشه.")
print("مرحله ۳: وقتی آپلود تموم شد، اسم فایل رو (مثلاً room.jpg) پایین همین سلول بنویس و Enter بزن.")
print()

test_bytes = None
while test_bytes is None:
    entered = input("اسم فایل (یا مسیر کامل) عکس آپلودشده: ").strip()
    candidates = [entered, os.path.join("/content", entered)] if not os.path.isabs(entered) else [entered]
    for candidate in candidates:
        if os.path.exists(candidate):
            with open(candidate, "rb") as f:
                test_bytes = f.read()
            break
    if test_bytes is None:
        print(f"فایل پیدا نشد ({entered}) — مطمئن شو مرحلهٔ ۲ (drag & drop تو پنل فایل) واقعاً تموم شده، بعد دوباره امتحان کن.")

test_img = Image.open(io.BytesIO(test_bytes))
test_img = ImageOps.exif_transpose(test_img).convert("RGB")
if max(test_img.size) > 2400:
    scale = 2400 / max(test_img.size)
    test_img = test_img.resize((round(test_img.width * scale), round(test_img.height * scale)), Image.Resampling.LANCZOS)
buf = io.BytesIO()
test_img.save(buf, format="JPEG", quality=92)
test_b64 = "data:image/jpeg;base64," + base64.b64encode(buf.getvalue()).decode()

print(f"در حال تولید ({TEST_STYLE} / {TEST_MODEL})...")
result = generate_style(test_b64, style_name=TEST_STYLE, model=TEST_MODEL, seed=42)

fig, axes = plt.subplots(1, 2, figsize=(13, 6.5))
axes[0].imshow(base64_to_pil(test_b64)); axes[0].set_title("قبل"); axes[0].axis("off")
axes[1].imshow(base64_to_pil(result["image"])); axes[1].set_title(f"بعد ({TEST_STYLE}, {TEST_MODEL})"); axes[1].axis("off")
plt.tight_layout()
plt.show()


مرحله ۱: تو نوار سمت چپ Colab، روی آیکون پوشه 📁 کلیک کن.
مرحله ۲: عکس اتاق رو از کامپیوترت بکش و همون‌جا ول کن (drag & drop) تا آپلود بشه.
مرحله ۳: وقتی آپلود تموم شد، اسم فایل رو (مثلاً room.jpg) پایین همین سلول بنویس و Enter بزن.



KeyboardInterrupt: Interrupted by user

## ۱۲) سرور Flask



In [125]:
import threading as _threading
import secrets, hmac
from flask import Flask, request, jsonify
from werkzeug.exceptions import HTTPException

colab_app = Flask(__name__)
colab_app.config["MAX_CONTENT_LENGTH"] = 32 * 1024 * 1024
CONNECTION_KEY = secrets.token_urlsafe(32)

try:
    from google.colab import userdata as _colab_userdata
    OPENROUTER_API_KEY = _colab_userdata.get("OPENROUTER_API_KEY")
except Exception:
    OPENROUTER_API_KEY = os.environ.get("OPENROUTER_API_KEY")
    if not OPENROUTER_API_KEY:
        
        try:
            from kaggle_secrets import UserSecretsClient
            OPENROUTER_API_KEY = UserSecretsClient().get_secret("OPENROUTER_API_KEY")
        except Exception:
            pass

if not OPENROUTER_API_KEY:
    from getpass import getpass
    OPENROUTER_API_KEY = getpass(
        "OpenRouter API key for AI prompt enhancement (optional, hidden -- Enter to skip): "
    ).strip() or None
OPENROUTER_MODEL = os.environ.get("OPENROUTER_MODEL", "inclusionai/ling-3.0-flash-vl:free")


_ENHANCE_SYSTEM_PROMPT = (
    "You expand short interior-design prompts into vivid, concrete visual detail "
    "(materials, colors, lighting, furniture style) for an AI image generator, while "
    "keeping every specific item the user named. Keep the same scope and intent exactly "
    "-- never add a new room, structural changes, or objects the user didn't ask for. "
    "Reply with ONLY the rewritten prompt: STRICTLY 15-20 words, one sentence, no "
    "preamble, no quotes."
)


def enhance_prompt(text):
    if not OPENROUTER_API_KEY or not isinstance(text, str) or not text.strip():
        return text
    try:
        r = requests.post(
            "https://openrouter.ai/api/v1/chat/completions",
            headers={"Authorization": f"Bearer {OPENROUTER_API_KEY}"},
            json={
                "model": OPENROUTER_MODEL,
                "messages": [
                    {"role": "system", "content": _ENHANCE_SYSTEM_PROMPT},
                    {"role": "user", "content": text},
                ],
                "max_tokens": 60,  # ~20 words -- see _ENHANCE_SYSTEM_PROMPT comment on the CLIP budget
            },
            timeout=12,
        )
        r.raise_for_status()
        out = r.json()["choices"][0]["message"]["content"].strip().strip("\"'")
      
        if not out or len(out.split()) < max(6, len(text.split())):
            return text
        return out
    except Exception:
        return text


@colab_app.after_request
def add_cors(response):
    response.headers["Access-Control-Allow-Origin"] = "*"
    response.headers["Access-Control-Allow-Headers"] = "Content-Type, ngrok-skip-browser-warning, Authorization"
    response.headers["Access-Control-Allow-Methods"] = "GET, POST, PUT, DELETE, OPTIONS"
    return response


@colab_app.before_request
def check_auth():
    if request.method == "OPTIONS":
        resp = colab_app.make_response("")
        resp.headers["Access-Control-Allow-Origin"] = "*"
        resp.headers["Access-Control-Allow-Headers"] = "Content-Type, ngrok-skip-browser-warning, Authorization"
        resp.headers["Access-Control-Allow-Methods"] = "GET, POST, PUT, DELETE, OPTIONS"
        resp.status_code = 200
        return resp
    if not hmac.compare_digest(request.headers.get("Authorization", ""), "Bearer " + CONNECTION_KEY):
        return jsonify({"error": "Connection key required"}), 401


@colab_app.errorhandler(ValueError)
def invalid_input(exc):
    return jsonify({"error": str(exc)}), 400


@colab_app.errorhandler(Exception)
def request_error(exc):
    if isinstance(exc, HTTPException):
        return jsonify({"error": exc.description}), exc.code
    colab_app.logger.exception("Backend request failed")
    return jsonify({"error": f"{type(exc).__name__}: {exc}"}), 500


def payload():
    data = request.get_json(silent=True)
    if not isinstance(data, dict):
        raise ValueError("JSON object required")
    return data


def request_image(data):
    if data.get("image"):
        return pil_to_base64(base64_to_pil(data["image"]))
    key = data.get("image_id")
    if not isinstance(key, str) or key not in IMAGE_STORE:
        raise ValueError("Send image or a valid image_id from /upload; expired images must be uploaded again")
    return IMAGE_STORE[key]


def request_seed(data):
    value = data.get("seed", 42)
    if type(value) is not int or not 0 <= value < 2 ** 32:
        raise ValueError("seed must be an integer 0..4294967295")
    return value


def request_model(data):
    return data.get("model") if data.get("model") in ("fast", "quality") else "fast"


def finish(result):
    if "image" in result:
        result.update(remember_image(base64_to_pil(result["image"])))
    return jsonify(result)


@colab_app.get("/health")
def health():
    return jsonify({"status": "ok", "colab_connected": True, "mode": "colab",
                     "api_version": 2, "current_model": _current_model["name"]})


@colab_app.get("/capabilities")
def capabilities():
    return jsonify({
        "api_version": 2, "styles": list(STYLE_PROMPTS), "models": ["fast", "quality"],
        "current_model": _current_model["name"],
        "operations": ["style", "furnish", "detect", "edit", "delete", "add-object", "recolor", "texture", "generate-texture"],
        "textures": list(TEXTURE_PROMPTS),
        "selection": ["region_id", "mask", "bbox", "point", "points"],
        "coordinates": "normalized", "furnish_requires_selection": False,
        "output_mime_type": "image/png",
        "prompt_enhance": bool(OPENROUTER_API_KEY),
    })


@colab_app.post("/enhance-prompt")
def enhance_prompt_route():
    data = payload()
    return jsonify({"prompt": enhance_prompt(data.get("prompt", ""))})


@colab_app.post("/upload")
def upload():
    if "image" not in request.files:
        raise ValueError("image file required")
    im = base64_to_pil(base64.b64encode(request.files["image"].read()).decode())
    with MODEL_LOCK:
        return jsonify(remember_image(im))


@colab_app.post("/generate")
@colab_app.post("/colab-generate")
def generate():
    data = payload()
    with MODEL_LOCK:
        return finish(generate_style(
            request_image(data), style_name=data.get("style"), palette=data.get("palette"),
            custom_prompt=data.get("customPrompt"), colors_only=bool(data.get("colorsOnly")),
            extra_details=data.get("extraDetails"), model=request_model(data), seed=request_seed(data),
            draft=bool(data.get("draft"))))


@colab_app.post("/redesign-from-reference")
@colab_app.post("/colab-redesign-from-reference")
def redesign_from_reference_route():
    data = payload()
    with MODEL_LOCK:
        return finish(generate_style(
            request_image(data), reference_image=data.get("referenceImage"), palette=data.get("palette"),
            model=request_model(data), seed=request_seed(data), draft=bool(data.get("draft"))))


@colab_app.post("/detect-objects")
@colab_app.post("/colab-detect")
def detect_objects_route():
    data = payload()
    with MODEL_LOCK:
        return jsonify(detect_objects(request_image(data)))


@colab_app.post("/segment-point")
def segment_point_route():
    data = payload()
    with MODEL_LOCK:
        im = base64_to_pil(request_image(data))
        mask = selection_mask(im, {"point": data.get("point")})
        key, rid = image_key(im), _uuid.uuid4().hex
        regions = REGION_STORE.setdefault(key, {})
        if len(regions) >= 80:
            raise ValueError("Too many regions; run detection again")
        regions[rid] = {"label": "selected region", "mask": mask}
        while len(REGION_STORE) > MAX_REGION_SETS:
            REGION_STORE.popitem(last=False)
        return jsonify({"region_id": rid, "image_hash": key,
                         "mask": pil_to_base64(Image.fromarray(mask.astype("uint8") * 255)),
                         "width": im.width, "height": im.height, "mask_mime_type": "image/png"})


@colab_app.post("/segment-points")
def segment_points_route():
    data = payload()
    with MODEL_LOCK:
        im = base64_to_pil(request_image(data))
        mask = selection_mask(im, {"points": {"positive": data.get("positive"),
                                               "negative": data.get("negative", []),
                                               "bbox": data.get("bbox")}})
        return jsonify({"mask": pil_to_base64(Image.fromarray(mask.astype("uint8") * 255)),
                         "width": im.width, "height": im.height, "mask_mime_type": "image/png"})


@colab_app.post("/edit-object")
@colab_app.post("/colab-edit")
def edit_object_route():
    data = payload()
    with MODEL_LOCK:
        image_b64 = request_image(data)
        selection = resolve_selection(image_b64, data)
        return finish(edit_object(image_b64, data.get("object"), data.get("prompt"), selection,
                                   model=request_model(data), seed=request_seed(data)))


@colab_app.post("/delete-object")
@colab_app.post("/colab-delete")
def delete_object_route():
    data = payload()
    with MODEL_LOCK:
        image_b64 = request_image(data)
        selection = resolve_selection(image_b64, data)
        return finish(delete_object_lama(image_b64, selection))


@colab_app.post("/recolor-object")
@colab_app.post("/colab-recolor")
def recolor_object_route():
    data = payload()
    with MODEL_LOCK:
        image_b64 = request_image(data)
        selection = resolve_selection(image_b64, data)
        return finish(recolor_object(image_b64, selection, data.get("color"), data.get("strength", .85)))


@colab_app.post("/apply-texture")
def apply_texture_route():
    data = payload()
    texture = data.get("texture")
    if not texture:
        raise ValueError("texture image required")
    with MODEL_LOCK:
        image_b64 = request_image(data)
        selection = resolve_selection(image_b64, data)
        return finish(apply_texture(image_b64, selection, pil_to_base64(base64_to_pil(texture)),
                                     data.get("opacity", .85)))


@colab_app.post("/generate-texture")
def generate_texture_route():
    data = payload()
    texture_name = data.get("texture")
    if not texture_name:
        raise ValueError("texture name required")
    with MODEL_LOCK:
        image_b64 = request_image(data)
        selection = resolve_selection(image_b64, data)
        return finish(generate_texture(image_b64, selection, texture_name,
                                        request_model(data), request_seed(data)))


@colab_app.post("/furnish-room")
@colab_app.post("/colab-furnish")
def colab_furnish():
    data = payload()
    with MODEL_LOCK:
        return finish(furnish_room(request_image(data), data.get("prompt"), data.get("selection"),
                                    request_model(data), request_seed(data)))


@colab_app.post("/preview-styles")
@colab_app.post("/colab-preview")
def preview_styles_route():
    data = payload()
    with MODEL_LOCK:
        return jsonify(generate_all_previews(request_image(data), data.get("palette"), data.get("styles"),
                                              request_model(data), request_seed(data), draft=bool(data.get("draft"))))


@colab_app.post("/add-object")
@colab_app.post("/colab-add-object")
def add_object_route():
    data = payload()
    room_image = data.get("room_image")
    object_image = data.get("object_image")
    if not room_image or not object_image:
        raise ValueError("room_image and object_image required")
    room_b64 = pil_to_base64(base64_to_pil(room_image))
   
    obj_rgba = base64_to_pil_rgba(object_image)
    with MODEL_LOCK:
        return finish(place_library_object(room_b64, pil_to_base64(obj_rgba), data.get("prompt", ""),
                                            data.get("selection"), request_model(data), request_seed(data)))


print(" Flask app defined —", len(list(colab_app.url_map.iter_rules())), "routes registered")


OpenRouter API key for AI prompt enhancement (optional, hidden -- Enter to skip):  ········


 Flask app defined — 27 routes registered


## ۱۳)اجرا و اتصال 

In [126]:
from getpass import getpass
from pyngrok import ngrok
from werkzeug.serving import make_server
import logging

logging.getLogger("pyngrok").setLevel(logging.CRITICAL)


if "server" in globals():
    try:
        server.shutdown()
    except Exception:
        pass

ngrok.set_auth_token(getpass("Your ngrok authtoken (hidden): "))

server = make_server("127.0.0.1", 7860, colab_app, threaded=True)
_threading.Thread(target=server.serve_forever, daemon=True).start()

try:
    tunnel = ngrok.connect(7860)
except Exception:
    raise RuntimeError("Tunnel failed. Check your own ngrok authtoken and account.") from None

print("BACKEND_URL:", tunnel.public_url)
print("CONNECTION_KEY:", CONNECTION_KEY)
print("این دو مقدار رو تو مودال «Connect AI Backend» فرانت وارد کن. رانتایم رو زنده نگه دار.")


Your ngrok authtoken (hidden):  ········


BACKEND_URL: https://81dc-35-236-134-60.ngrok-free.app
CONNECTION_KEY: QePjT9qmAqd_r3c6Aql7cRiSrUmZXJb5jWSAsGkPYxQ
این دو مقدار رو تو مودال «Connect AI Backend» فرانت وارد کن. رانتایم رو زنده نگه دار.


INFO:werkzeug:127.0.0.1 - - [22/Sep/2026 08:23:07] "OPTIONS /capabilities HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [22/Sep/2026 08:23:08] "GET /capabilities HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [22/Sep/2026 08:23:09] "GET /capabilities HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [22/Sep/2026 08:23:14] "OPTIONS /upload HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [22/Sep/2026 08:23:15] "POST /upload HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [22/Sep/2026 08:23:17] "OPTIONS /capabilities HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [22/Sep/2026 08:23:17] "GET /capabilities HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [22/Sep/2026 08:25:12] "OPTIONS /enhance-prompt HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [22/Sep/2026 08:25:12] "POST /enhance-prompt HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [22/Sep/2026 08:25:13] "OPTIONS /add-object HTTP/1.1" 200 -


Switching model: fast/style -> fast/inpaint (unloading previous one)...
Loading SD1.5 inpainting (Fast mode)...


Loading pipeline components...:   0%|          | 0/6 [00:00<?, ?it/s]

An error occurred while trying to fetch /content/interior_v2/models/models--stable-diffusion-v1-5--stable-diffusion-inpainting/snapshots/8a4288a76071f7280aedbdb3253bdb9e9d5d84bb/vae: Error no file named diffusion_pytorch_model.safetensors found in directory /content/interior_v2/models/models--stable-diffusion-v1-5--stable-diffusion-inpainting/snapshots/8a4288a76071f7280aedbdb3253bdb9e9d5d84bb/vae.
Defaulting to unsafe serialization. Pass `allow_pickle=False` to raise an error instead.
An error occurred while trying to fetch /content/interior_v2/models/models--stable-diffusion-v1-5--stable-diffusion-inpainting/snapshots/8a4288a76071f7280aedbdb3253bdb9e9d5d84bb/unet: Error no file named diffusion_pytorch_model.safetensors found in directory /content/interior_v2/models/models--stable-diffusion-v1-5--stable-diffusion-inpainting/snapshots/8a4288a76071f7280aedbdb3253bdb9e9d5d84bb/unet.
Defaulting to unsafe serialization. Pass `allow_pickle=False` to raise an error instead.
You have disabled 

 SD1.5 inpainting ready


  0%|          | 0/18 [00:00<?, ?it/s]

INFO:werkzeug:127.0.0.1 - - [22/Sep/2026 08:25:37] "POST /add-object HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [22/Sep/2026 08:25:38] "OPTIONS /capabilities HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [22/Sep/2026 08:25:39] "GET /capabilities HTTP/1.1" 200 -
